# 01 — Generate synthetic longitudinal multimodal cohort

This notebook generates the synthetic dataset that the rest of the project uses.
The generator simulates 250 participants × 90 days of wearable, smartphone,
EMA-survey, and environmental signals with realistic between- and
within-person variance and informative missingness.

**No real patient data is involved.** See `paper/data_card.md` for details.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lhfm.data.synthetic_generator import SyntheticCohortGenerator, GeneratorConfig
from lhfm.data.validation import validate_synthetic_dataframe

pd.set_option('display.max_columns', 30)

In [ ]:
cfg = GeneratorConfig(n_participants=250, n_days=90, seed=42)
gen = SyntheticCohortGenerator(cfg)
df = gen.generate()
print(df.shape)
df.head()

### Validate

In [ ]:
report = validate_synthetic_dataframe(df)
print('ok      :', report.ok)
print('warnings:', report.warnings)
print('summary :', report.summary)

### Inspect distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col in zip(axes.flat, ['sleep_duration', 'hrv_rmssd', 'resting_hr',
                               'survey_mood', 'aqi', 'heat_index']):
    df[col].dropna().hist(bins=40, ax=ax)
    ax.set_title(col)
fig.tight_layout(); plt.show()

### Plot a single participant trajectory

In [ ]:
pid = df['participant_id'].iloc[0]
sub = df[df['participant_id'] == pid].copy()
sub['date'] = pd.to_datetime(sub['date'])

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
axes[0].plot(sub.date, sub.sleep_duration); axes[0].set_ylabel('sleep (h)')
axes[1].plot(sub.date, sub.hrv_rmssd);      axes[1].set_ylabel('HRV (ms)')
axes[2].plot(sub.date, sub.survey_mood);    axes[2].set_ylabel('mood (1-7)')
axes[3].plot(sub.date, sub.heat_index);     axes[3].set_ylabel('heat idx (°C)')
for ax in axes: ax.grid(alpha=0.3)
fig.suptitle(f'participant {pid}'); fig.tight_layout(); plt.show()

### Save

In [ ]:
gen.save(df, Path('..') / 'data' / 'synthetic' / 'cohort.csv')
print('saved.')